In [13]:
import json
from pathlib import Path
from collections import defaultdict
import ranx 
from ranx import Qrels, Run, evaluate, fuse
import orjson
# --- 1. Setup Paths ---
base_dir1 = Path("/home/ucloud/BioASQ13B/phaseA-reranker/refactored-trainer/outputs")

golden_files = [
    "/home/ucloud/BioASQ13B/data/val_data/13B1_golden.json",
    "/home/ucloud/BioASQ13B/data/val_data/13B2_golden.json",
    "/home/ucloud/BioASQ13B/data/val_data/13B3_golden.json",
    "/home/ucloud/BioASQ13B/data/val_data/13B4_golden.json",
]

# Set your threshold (ranx stores scores as 0.0 to 1.0)
SCORE_THRESHOLD = 0.64

# --- 2. Load and Combine BioASQ Golden Files into Qrels ---
print("Loading Golden Files...")
qrels_dict = defaultdict(dict)

for file_path in golden_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        
        for q in data.get("questions", []):
            qid = str(q["id"])
            
            for doc in q.get("documents", []):
                # BioASQ URLs look like: http://www.ncbi.nlm.nih.gov/pubmed/12345678
                clean_doc_id = str(doc).rstrip('/').split('/')[-1]
                qrels_dict[qid][clean_doc_id] = 1

qrels = Qrels(dict(qrels_dict))
print(f"Loaded {len(qrels_dict)} unique queries into Qrels.\n")

# --- 3. Filter and Gather Prediction Files ---
print(f"Scanning models and filtering for map-bioasq@10 > {SCORE_THRESHOLD}...")
prediction_files = []

for subdir1 in base_dir1.iterdir():
    for subdir in subdir1.iterdir():
    
        results_path = subdir / "ranx_results.json"
        
        if results_path.exists():
            with open(results_path, "r", encoding="utf-8") as f:
                try:
                    res_data = json.load(f)
                    
                    # Handle cases where results are nested under "total"
                    if "total" in res_data:
                        score = res_data["total"].get("map-bioasq@10", 0.0)
                    else:
                        score = res_data.get("map-bioasq@10", 0.0)
                    
                    # Fallback just in case your results were saved as percentages (e.g., 59.4 instead of 0.594)
                    actual_threshold = SCORE_THRESHOLD * 100 if score > 1.0 else SCORE_THRESHOLD
                    
                    if score > actual_threshold:
                        # Find the corresponding predictions file
                        pred_path =  (Path("refactored-trainer") / "batch01" / "runs").glob(f"{subdir.name}*.json") # subdir / "predictions" / "predictions.json"
                        pred_path = list(pred_path)[0]
                        if pred_path.exists():
                            prediction_files.append(str(pred_path))
                            print(f"✅ [KEEP] {subdir.name} (Score: {score:.4f})")
                        else:
                            print(f"⚠️ [WARNING] High score but missing predictions.json for {subdir.name}")
                    else:
                        print(f"❌ [SKIP] {subdir.name} (Score: {score:.4f})")
                        
                except json.JSONDecodeError:
                    print(f"⚠️ [WARNING] Corrupted ranx_results.json in {subdir.name}")
        else:
            # Silently skip subdirectories that don't have results (could be checkpoints, logs, etc.)
            pass

    



print(f"\nFound {len(prediction_files)} models that met the criteria.")
if not prediction_files:
    raise ValueError("No models met the threshold! Try lowering the threshold or check your results files.")

# --- 4. Load and Fuse Runs ---
print("\nLoading runs into ranx...")
ranx_runs = [Run.from_file(run) for run in prediction_files]

print("Applying Reciprocal Rank Fusion (RRF)...")
fused_run = fuse(ranx_runs, method="rrf")

print("Fusing runs using RRF...")
with open("fused_rrf_predictions.json", "wb") as f:
    f.write(orjson.dumps(dict(fused_run)))


# --- 5. Evaluate ---
metrics = [
    "ndcg@5",
    "mrr",
    "recall@10",
    "recall@100",
    "map@10",
    "map-bioasq@10", 
]

print("Evaluating Fused Run...")
results = evaluate(qrels, fused_run, metrics=metrics)

print("\n=== RRF Ensemble Results ===")
for metric, score in results.items():
    print(f"{metric}: {score:.4f}")

Loading Golden Files...
Loaded 340 unique queries into Qrels.

Scanning models and filtering for map-bioasq@10 > 0.64...
❌ [SKIP] michiyasunaga-BioLinkBERT-base-E2-S1-Mpairwise-FullDataTrue (Score: 0.6178)
❌ [SKIP] michiyasunaga-BioLinkBERT-base-E2-S1-Mpairwise-FullDataTrue--BasicV2 (Score: 0.5339)
✅ [KEEP] nvidia-llama-nemotron-rerank-1b-v2-E2-S4-Mmulti_neg_pairwise-Linfonce-FullData (Score: 0.9995)
❌ [SKIP] microsoft-BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-E2-S1-Mpairwise-FullDataTrue--BasicV2 (Score: 0.5586)
❌ [SKIP] microsoft-BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext-E2-S1-Mpairwise-FullDataTrue (Score: 0.6153)
❌ [SKIP] pritamdeka-S-PubMedBert-MS-MARCO-E2-S1-Mpairwise-FullDataTrue (Score: 0.5839)
❌ [SKIP] monologg-biobert_v1.1_pubmed-E2-S1-Mpairwise-FullDataTrue (Score: 0.6053)
❌ [SKIP] michiyasunaga-BioLinkBERT-large-E2-S1-Mpairwise-FullDataTrue (Score: 0.5781)
❌ [SKIP] cross-encoder-ms-marco-MiniLM-L-6-v2-E2-S1-Mpairwise-FullDataTrue (Score: 0.5944)
❌ [SKIP]

Applying Reciprocal Rank Fusion (RRF)...
Fusing runs using RRF...
Evaluating Fused Run...


AssertionError: Qrels and Run query ids do not match

In [4]:

import ranx
import inspect

# ranx stores its metric functions in a dictionary/module. 
# The easiest way to check is to see if your script runs without the error!
try:
    from ranx import Qrels, Run, evaluate
    
    # Create dummy data just to trigger the metric check
    qrels = Qrels({"q1": {"d1": 1}})
    run = Run({"q1": {"d1": 0.9}})
    
    # Try the BioASQ metric
    evaluate(qrels, run, "map-bioasq@10")
    print("✅ SUCCESS! The custom map-bioasq@10 metric is installed and recognized.")
except Exception as e:
    print(f"❌ FAILED: {e}")

✅ SUCCESS! The custom map-bioasq@10 metric is installed and recognized.


In [ ]:
# !uv sync


Resolved 158 packages in 24ms
Uninstalled 52 packages in 870ms
 - asttokens==3.0.1
 - asyncio==4.0.0
 - bitsandbytes==0.49.2
 - comm==0.2.3
 - cut-cross-entropy==25.1.1
 - debugpy==1.8.20
 - decorator==5.2.1
 - deepspeed==0.18.8
 - diffusers==0.37.0
 - docstring-parser==0.17.0
 - einops==0.8.2
 - executing==2.2.1
 - flagembedding==1.3.5
 - hf-transfer==0.1.9
 - hjson==3.1.0
 - importlib-metadata==8.7.1
 - ipykernel==7.2.0
 - ipython==9.11.0
 - ipython-pygments-lexers==1.1.1
 - jedi==0.19.2
 - jupyter-client==8.8.0
 - jupyter-core==5.9.1
 - matplotlib-inline==0.2.1
 - msgpack==1.1.2
 - msgspec==0.20.0
 - nest-asyncio==1.6.0
 - ninja==1.13.0
 - parso==0.8.6
 - pexpect==4.9.0
 - pillow==12.1.1
 - prompt-toolkit==3.0.52
 - ptyprocess==0.7.0
 - pure-eval==0.2.3
 - py-cpuinfo==9.0.0
 - pyzmq==27.1.0
 - scikit-learn==1.8.0
 - sentence-transformers==5.3.0
 - sentencepiece==0.2.1
 - stack-data==0.6.3
 - threadpoolctl==3.6.0
 - torchao==0.16.0
 - torchvision==0.25.0
 - tornado==6.5.5
 - traitlet

In [4]:

# --- 3. Load and Fuse Runs ---
print("Loading runs into ranx...")
ranx_runs = [Run.from_file(run) for run in prediction_files]

print("Fusing runs using RRF...")
fused_run = fuse(ranx_runs, method="rrf")

# Optional: Save the fused run if you need it for later
# fused_run.save("fused_rrf_predictions.json")


Loading runs into ranx...
Fusing runs using RRF...


AssertionError: Only one run provided

In [ ]:

# --- 4. Evaluate ---
print("Loading Qrels (ground truth)...")
qrels = Qrels.from_file(qrels_path)

# Define the metrics you care about (these are common for BioASQ/IR)
metrics = ["map@10", "mrr", "ndcg@10", "recall@100"]

print("Evaluating...")
results = evaluate(qrels, fused_run, metrics=metrics)

# Print the final dictionary of results
print("\n=== Evaluation Results ===")
for metric, score in results.items():
    print(f"{metric}: {score:.4f}")